In [ ]:
from pathlib import Path

RUN = "runs/plan-a-nested"

# Works when the kernel starts in the repository root or in notebooks/.
ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "stratscout").is_dir() and (p / "AGENTS.md").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CV-DL repository or its notebooks folder.")

run_dir = (ROOT / RUN).resolve()
if not run_dir.is_dir():
    raise FileNotFoundError(
        f"Run folder does not exist: {run_dir}\n"
        "Change RUN to an existing default-mode run, for example "
        "'runs/plan-a-nested-20261116', or create the default run.\n"
        "From the repository root, with the .venv environment active, run:\n"
        "python -m stratscout tactic-baseline "
        "--manifest data/processed/datasets/2026-11-16/train/manifest.csv "
        f'--output "{RUN}" --exclude-uncertain'
    )
print(f"Reading saved results from {run_dir}")

# Plan A results

1. Select the project's **.venv** Python kernel. If it is missing, install `ipykernel` in that environment.
2. Set `RUN` in the first cell to an existing default-mode run, then run the cells from top to bottom.
3. Keep the figures on screen; this notebook does not write images, change results or train a model.

This view shows every validation match and seeds 0, 1 and 2 from the default Plan A run. Each section draws one figure per validation match.
It works for a nested run (three or more training matches) and for the older two-match cross-fold run.
It reads saved JSON files only. It never opens clips, checkpoints or the test split.
These are development scores, not independent test results: the same matches were used to choose the settings.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

plt.switch_backend("module://matplotlib_inline.backend_inline")
plt.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "figure.dpi": 110,
    }
)

config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
mode = config.get("epoch_mode")
if config.get("split") != "train" or mode not in ("nested", "cross-fold"):
    raise ValueError("Choose a completed nested or cross-fold run made from the training manifest.")
mode_name = mode.capitalize()

folds = {}
for path in sorted(run_dir.glob("fold-*.json")):
    if path.stem.removeprefix("fold-").isdigit():
        saved = json.loads(path.read_text(encoding="utf-8"))
        folds[saved["fold"]] = saved
if len(folds) < 2 or set(folds) != set(range(1, len(folds) + 1)):
    raise ValueError("This notebook needs every fold-N.json of a completed run, at least two.")
if (mode == "cross-fold") != (len(folds) == 2):
    raise ValueError("Cross-fold runs have two validation matches; nested runs have three or more.")

runs_by_fold = {number: {r["seed"]: r for r in fold["runs"]} for number, fold in folds.items()}
for number, runs in runs_by_fold.items():
    if not {0, 1, 2}.issubset(runs):
        raise ValueError(f"Fold {number} is missing seeds 0, 1 or 2; use a default three-seed run.")
    if any(r["epoch_selection"] != mode or r.get("optimistic") is None for r in runs.values()):
        raise ValueError(f"Both {mode} and optimistic comparison scores are required.")

samples = json.loads((run_dir / "samples.json").read_text(encoding="utf-8"))
if any(sample["split"] != "train" for sample in samples):
    raise ValueError("The saved samples must all belong to the training split.")
match_folders = {sample["match_id"]: sample["match_folder"] for sample in samples}
names = {
    "furia": "Furia vs Falcons",
    "spirit": "Spirit vs Falcons",
    "aurora": "Aurora vs Vitality",
    "mouz": "MOUZ vs BetBoom",
}
match_names = {
    match_id: names.get(folder.split("-")[0], folder.replace("-", " ").title())
    for match_id, folder in match_folders.items()
}
fold_names = {number: match_names[fold["validation_match_id"]] for number, fold in folds.items()}

classes = config["classes"]
class_names = [label.replace("_", " ").capitalize() for label in classes]
seeds = sorted(runs_by_fold[1])
if any(sorted(runs) != seeds for runs in runs_by_fold.values()):
    raise ValueError("Every fold must have the same seeds for this comparison.")


def show_confusion(matrix, title):
    fig, ax = plt.subplots(figsize=(8.4, 6.8), layout="constrained")
    heatmap = ax.imshow(matrix, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
    ax.set_yticks(range(len(classes)), class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    for row in range(len(classes)):
        for column in range(len(classes)):
            ax.text(
                column,
                row,
                str(int(matrix[row, column])),
                ha="center",
                va="center",
                fontsize=12,
                color="white" if matrix[row, column] > matrix.max() / 2 else "black",
            )
    colourbar = fig.colorbar(heatmap, ax=ax, label="Number of predictions")
    colourbar.locator = MaxNLocator(integer=True)
    colourbar.update_ticks()
    plt.show()
    plt.close(fig)


print("Epoch choice:", mode)
print("Class order:", ", ".join(class_names))
print("Validation matches:", ", ".join(fold_names.values()))
print("Seeds:", seeds)
print("Excluded clips:", config["excluded_clip_ids"] or "none")

### Confusion matrix for seed 0, one per validation match

Counts on the diagonal are correct predictions. Off-diagonal counts show which classes are confused. Rows are true classes; columns are predicted classes.

In [ ]:
for number in folds:
    matrix = np.asarray(runs_by_fold[number][0]["metrics"]["confusion_matrix"])
    show_confusion(
        matrix,
        f"{fold_names[number]}: {mode} confusion\nSeed 0; {matrix.sum()} validation clips",
    )

### Confusion matrix summed over seeds, one per validation match

Look for mistakes that repeat across seeds. Each clip is counted once per seed, so these are prediction counts, not extra clips. Rows are true classes; columns are predicted classes.

In [ ]:
for number in folds:
    matrix = np.sum(
        [run["metrics"]["confusion_matrix"] for run in runs_by_fold[number].values()],
        axis=0,
    )
    show_confusion(
        matrix,
        f"{fold_names[number]}: {mode} confusion\n"
        f"Sum over {len(runs_by_fold[number])} seeds; {matrix.sum()} predictions",
    )

### F1 for each class, one figure per validation match

Compare the main scores with the optimistic scores selected on the same validation match. Bars average the saved seed results. The count under each class is the number of validation clips; small counts make the score less reliable.

In [ ]:
for number, fold in folds.items():
    main_f1 = np.mean(
        [
            [run["metrics"]["per_class"][label]["f1"] for label in classes]
            for run in runs_by_fold[number].values()
        ],
        axis=0,
    )
    optimistic_f1 = np.mean(
        [
            [run["optimistic"]["metrics"]["per_class"][label]["f1"] for label in classes]
            for run in runs_by_fold[number].values()
        ],
        axis=0,
    )
    positions = np.arange(len(classes))
    labels = [
        f"{name}\n(n={fold['validation_counts'][label]})"
        for label, name in zip(classes, class_names, strict=True)
    ]
    fig, ax = plt.subplots(figsize=(10, 5.5), layout="constrained")
    main_bars = ax.bar(positions - 0.19, main_f1, 0.38, color="#0072B2", label=mode_name)
    optimistic_bars = ax.bar(
        positions + 0.19,
        optimistic_f1,
        0.38,
        color="#D55E00",
        label="Optimistic (selected on validation)",
    )
    ax.bar_label(main_bars, fmt="%.2f", padding=3, fontsize=9)
    ax.bar_label(optimistic_bars, fmt="%.2f", padding=3, fontsize=9)
    ax.set_xticks(positions, labels)
    ax.set_ylim(0, 1.1)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_xlabel("Class (number of validation clips)")
    ax.set_ylabel("Mean F1")
    ax.set_title(
        f"{fold_names[number]}: per-class F1, mean over {len(runs_by_fold[number])} seeds"
    )
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(loc="upper left")
    plt.show()
    plt.close(fig)

## Training curves

The main fit records training loss at every epoch, but evaluates validation **only at the scored epoch**.
The dashed validation curves below come from a separate search on the scored match itself (`*-optimistic-history.json`).
They are a comparison, not extra measurements, and they did not choose the scored epoch.
In a nested run the dotted line is the curve that did choose it: mean macro-F1 over the inner validation matches (`*-selection-history.json`).
The final score is shown as a diamond, with a vertical line at its scored epoch.
Training loss uses class weights; validation loss is unweighted, so their values are not directly comparable.

### Loss and validation F1, one figure per validation match and seed

Look for training loss falling while validation stops improving. The vertical line marks the scored epoch: chosen on the other training matches in a nested run, or borrowed from the other fold in a cross-fold run. The diamonds are the measurements at that epoch.

In [ ]:
for number in folds:
    for seed in seeds:
        run = runs_by_fold[number][seed]
        stem = f"fold-{number}-seed-{seed}"
        history = json.loads((run_dir / f"{stem}-history.json").read_text(encoding="utf-8"))
        search_history = json.loads(
            (run_dir / f"{stem}-optimistic-history.json").read_text(encoding="utf-8")
        )
        scored_epoch = run["evaluation_epoch"]
        scored = next(row for row in history if row["epoch"] == scored_epoch)
        epochs = [row["epoch"] for row in history]
        search_epochs = [row["epoch"] for row in search_history]
        last_epoch = max(scored_epoch, max(search_epochs))
        fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
        loss_ax.plot(
            epochs,
            [row["train_loss"] for row in history],
            color="#0072B2",
            label=f"{mode_name} training loss",
        )
        loss_ax.plot(
            search_epochs,
            [row["validation_loss"] for row in search_history],
            color="#D55E00",
            linestyle="--",
            alpha=0.85,
            label="Search on the scored match: validation loss",
        )
        loss_ax.scatter(
            scored_epoch,
            scored["validation_loss"],
            color="#D55E00",
            edgecolor="black",
            marker="D",
            s=65,
            zorder=5,
            label=f"{mode_name} validation at scored epoch",
        )
        f1_ax.plot(
            search_epochs,
            [row["validation_macro_f1"] for row in search_history],
            color="#009E73",
            linestyle="--",
            alpha=0.85,
            label="Search on the scored match: macro-F1",
        )
        if mode == "nested":
            selection = json.loads(
                (run_dir / f"{stem}-selection-history.json").read_text(encoding="utf-8")
            )
            last_epoch = max(last_epoch, selection[-1]["epoch"])
            f1_ax.plot(
                [row["epoch"] for row in selection],
                [row["mean_validation_macro_f1"] for row in selection],
                color="#CC79A7",
                linestyle=":",
                linewidth=2,
                label="Inner matches: mean macro-F1 (chose the epoch)",
            )
        f1_ax.scatter(
            scored_epoch,
            scored["validation_macro_f1"],
            color="#009E73",
            edgecolor="black",
            marker="D",
            s=65,
            zorder=5,
            label=f"{mode_name} validation at scored epoch",
        )
        for ax in (loss_ax, f1_ax):
            ax.axvline(
                scored_epoch,
                color="#333333",
                linestyle="-.",
                linewidth=1.4,
                label=f"Scored epoch {scored_epoch}",
            )
            ax.set_xlabel("Epoch")
            ax.set_xlim(0, last_epoch + 1)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(alpha=0.2)
            ax.legend(loc="upper right")
        loss_ax.set_title("Training and validation loss")
        loss_ax.set_ylabel("Cross-entropy loss")
        loss_ax.set_ylim(bottom=0)
        f1_ax.set_title("Validation macro-F1")
        f1_ax.set_ylabel("Macro-F1")
        f1_ax.set_ylim(0, 1)
        fig.suptitle(f"{fold_names[number]}: seed {seed}, saved training history", fontsize=15)
        plt.show()
        plt.close(fig)

### Macro-F1 across seeds, one figure per validation match

Look at how much the score changes with initialisation. The dashed line is the majority-class guess chosen from the training matches. Seed variation does not measure uncertainty on unseen matches.

In [ ]:
for number, fold in folds.items():
    scores = [runs_by_fold[number][seed]["metrics"]["macro_f1"] for seed in seeds]
    majority = fold["majority"]["macro_f1"]
    majority_class = fold["majority_class"].replace("_", " ").capitalize()
    fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
    ax.plot(
        seeds, scores, marker="o", markersize=8, color="#0072B2", label=f"{mode_name} macro-F1"
    )
    ax.axhline(
        majority,
        linestyle="--",
        color="#555555",
        label=f"Majority: {majority_class} (F1={majority:.3f})",
    )
    for seed, score in zip(seeds, scores, strict=True):
        ax.annotate(
            f"{score:.3f}", (seed, score), xytext=(0, 10), textcoords="offset points", ha="center"
        )
    ax.set_xticks(seeds)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Random seed")
    ax.set_ylabel("Validation macro-F1")
    ax.set_title(f"{fold_names[number]}: {mode} macro-F1 by seed")
    ax.grid(axis="y", alpha=0.2)
    ax.legend(loc="upper right")
    plt.show()
    plt.close(fig)

## Keep in mind

These are scores on hand-cut clips from the development matches. A nested run keeps each validation match out of its own model and epoch choice, but the settings were still chosen while looking at these matches, and a handful of matches is a small sample. A two-match cross-fold run also reuses both matches: each validation match trained the other fold's model. Neither these plots nor the seed spread establish performance on an unseen match. NaVi and Vitality-Falcons remain held out.

Small classes have very few clips. The sum of confusion matrices counts repeated predictions of the same clips, and the per-class bars are means of seed-level F1 scores, not F1 calculated from a pooled matrix.